# NACLIP on Colab GPU from VS Code
Keep editing this local checkout with your agents. Select **Kernel > Colab > New Colab Server > GPU** first.

Upload the local code folders/files using **Upload to Colab**, including `tools`. Changes require another upload. See `COLAB.md`.
NACLIP is training-free: this notebook verifies inference and optionally evaluates a prepared dataset.


In [ ]:
from pathlib import Path
import subprocess
import sys

def run_remote(command, **kwargs):
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, **kwargs)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, command)

PROJECT = Path('/content/NACLIP')
PYTHON = Path('/content/naclip-env/bin/python')
if not Path('/content').is_dir():
    raise RuntimeError('Select a Colab kernel before running these cells.')
run_remote(['nvidia-smi'])
if not (PROJECT / 'tools/colab_setup.py').is_file():
    raise FileNotFoundError(f'Upload the checkout including tools to {PROJECT}, or adjust PROJECT.')
print('Remote project:', PROJECT)


## Install isolated dependencies
Run once per new runtime. Downloads can take several minutes. The final import check must succeed before continuing.

In [ ]:
run_remote([sys.executable, str(PROJECT / 'tools/colab_setup.py')], cwd=PROJECT)


## GPU inference check
Downloads the pretrained CLIP weights on first use and checks finite output on a synthetic image. This does not measure segmentation quality.

In [ ]:
run_remote([str(PYTHON), '-u', 'tools/colab_smoke.py'], cwd=PROJECT)


## Optional: mount Drive
Run only if your data/results live on Drive. Complete Google authorization yourself.

In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


## Optional: evaluate a dataset
Set the paths to an already prepared dataset and a persistent results folder. VOC21 expects JPEGImages, SegmentationClass, and ImageSets/Segmentation/val.txt under DATA_ROOT. Choose another upstream config for another benchmark. Local Windows paths are unavailable here.

In [ ]:
RUN_EVALUATION = False
CONFIG = 'configs/cfg_voc21.py'
DATA_ROOT = '/content/drive/MyDrive/datasets/VOCdevkit/VOC2012'
WORK_DIR = '/content/drive/MyDrive/NACLIP-results/voc21'
if RUN_EVALUATION:
    if WORK_DIR.startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Mount Drive first to persist evaluation results.')
    run_remote([str(PYTHON), 'tools/colab_eval.py', '--config', CONFIG,
                    '--data-root', DATA_ROOT, '--work-dir', WORK_DIR], cwd=PROJECT)
else:
    print('Evaluation disabled. Set dataset paths and RUN_EVALUATION=True when ready.')
